In [ ]:
import os
import subprocess
import pandas as pd
import numpy as np
import pickle
from sklearn.metrics import (matthews_corrcoef, accuracy_score, confusion_matrix, 
                             precision_score, recall_score, f1_score, 
                             roc_auc_score, average_precision_score)

# ================= 配置区域 =================
SEEDS = [42, 2023, 1234, 567, 100]
FOLDER_TEMPLATE = "split_seed_{}"    # 文件夹模板
OUTPUT_FILE = "blast_baseline_internal_pkl_metrics.csv"
# ===========================================

def run_command(cmd):
    """执行 Shell 命令"""
    try:
        subprocess.run(cmd, shell=True, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    except subprocess.CalledProcessError as e:
        print(f"Error executing: {cmd}")

def load_labels_from_pickle(pkl_path):
    """
    从 .pkl 文件加载标签。
    假设 pkl 内容是一个列表，每个元素是字典 {'id': ..., 'label': ...}
    或者 pkl 内容本身就是字典 {id: label}
    """
    if not os.path.exists(pkl_path):
        print(f"❌ Error: Pickle file not found: {pkl_path}")
        return {}, []

    try:
        with open(pkl_path, 'rb') as f:
            data = pickle.load(f)
        
        labels_map = {}
        ids_list = []
        
        # 情况 A: Data 是 list of dicts (最常见)
        if isinstance(data, list):
            for item in data:
                # 确保 ID 转为字符串，防止 int/str 不匹配
                # 假设键名是 'id' 和 'label'，如果是 'Entry_ID' 请修改
                seq_id = str(item.get('id', item.get('Entry_ID', '')))
                label = int(item.get('label', item.get('Label', 0)))
                if seq_id:
                    labels_map[seq_id] = label
                    ids_list.append(seq_id)
                    
        # 情况 B: Data 是 dict {id: label}
        elif isinstance(data, dict):
            for k, v in data.items():
                seq_id = str(k)
                label = int(v)
                labels_map[seq_id] = label
                ids_list.append(seq_id)
                
        return labels_map, ids_list

    except Exception as e:
        print(f"❌ Error loading pickle {pkl_path}: {e}")
        return {}, []

def calculate_full_metrics(y_true, y_pred, y_scores):
    """计算全套指标"""
    if not y_true: return {}
    
    # 混淆矩阵
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    
    # 基础指标
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0) # Sensitivity
    f1 = f1_score(y_true, y_pred, zero_division=0)
    mcc = matthews_corrcoef(y_true, y_pred)
    
    # 特异性 Specificity
    sp = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    # 高级指标 (AUC/AUPRC)
    try:
        if len(np.unique(y_true)) > 1:
            auc = roc_auc_score(y_true, y_scores)
            auprc = average_precision_score(y_true, y_scores)
        else:
            auc, auprc = 0.5, 0.0
    except:
        auc, auprc = 0.0, 0.0

    return {
        "Sens.": rec, "Spec.": sp, "Prec": prec, 
        "MCC": mcc, "AUROC": auc, "AUPRC": auprc,
        "Acc": acc, "F1": f1
    }

def main():
    all_metrics = []
    print(f"🚀 Starting BLAST Baseline using Pickle Labels (Seeds: {SEEDS})...")

    for seed in SEEDS:
        folder = FOLDER_TEMPLATE.format(seed)
        
        # 定义文件路径
        train_fasta = os.path.join(folder, "train.fasta")
        test_fasta = os.path.join(folder, "test.fasta")
        train_pkl = os.path.join(folder, "train_features.pkl")
        test_pkl = os.path.join(folder, "test_features.pkl")
        
        # 检查文件是否存在
        required_files = [train_fasta, test_fasta, train_pkl, test_pkl]
        if not all(os.path.exists(f) for f in required_files):
            print(f"⚠️ Skipping Seed {seed}: Missing one of the required files in {folder}")
            continue

        print(f"\n>>> Processing Seed {seed}...")
        
        # 1. 加载标签 (Ground Truth)
        # Train labels 用于查找 hit 是否为 VGIC
        train_labels_map, _ = load_labels_from_pickle(train_pkl)
        # Test labels 用于评估结果
        test_labels_map, test_ids = load_labels_from_pickle(test_pkl)
        
        print(f"   Loaded {len(train_labels_map)} train labels and {len(test_labels_map)} test labels.")

        # 2. 建立 BLAST 数据库 (使用 Train FASTA)
        db_name = f"temp_db_seed_{seed}"
        # -dbtype prot 表示蛋白质库
        run_command(f"makeblastdb -in {train_fasta} -dbtype prot -out {db_name}")

        # 3. 运行 BLASTP 搜索 (使用 Test FASTA 查询)
        out_file = f"blast_out_seed_{seed}.txt"
        # -max_target_seqs 1: 只取最佳匹配
        # -evalue 10: 宽松阈值以计算 AUC
        run_command(f"blastp -query {test_fasta} -db {db_name} -out {out_file} -outfmt \"6 qseqid sseqid evalue\" -max_target_seqs 1 -evalue 10 -num_threads 8")

        # 4. 解析 BLAST 结果
        blast_hits = {}    # {q_id: s_id}
        blast_evalues = {} # {q_id: evalue}
        
        if os.path.exists(out_file) and os.path.getsize(out_file) > 0:
            try:
                df = pd.read_csv(out_file, sep='\t', header=None, names=['qseqid', 'sseqid', 'evalue'])
                # 按 evalue 排序，去重取最优
                df = df.sort_values('evalue').drop_duplicates(subset='qseqid', keep='first')
                
                # 转换为字典，注意 ID 格式统一为字符串
                blast_hits = dict(zip(df['qseqid'].astype(str), df['sseqid'].astype(str)))
                blast_evalues = dict(zip(df['qseqid'].astype(str), df['evalue']))
            except Exception as e:
                print(f"   Error reading BLAST output: {e}")

        # 5. 评分与预测逻辑
        y_true = []
        y_pred = []
        y_scores = []

        # 遍历测试集中的每一条序列
        for q_id in test_ids:
            # 获取真实标签
            if q_id not in test_labels_map:
                continue # 理论上不应发生
            
            true_lbl = test_labels_map[q_id]
            y_true.append(true_lbl)
            
            # 默认预测 (No Hit)
            pred_lbl = 0
            score = 0.0
            
            # 检查是否有 BLAST Hit
            if q_id in blast_hits:
                s_id = blast_hits[q_id]
                e_val = blast_evalues[q_id]
                
                # 查找 Hit (s_id) 在训练集中的标签
                # 需要处理 ID 匹配问题，BLAST 可能会截断 ID (例如取第一个空格前)
                # 这里的逻辑是：直接查，查不到尝试模糊匹配
                found_train_label = train_labels_map.get(s_id)
                
                if found_train_label is None:
                    # 尝试处理 Uniprot ID 格式差异 (如 sp|ID|...)
                    # 尝试 1: 如果 s_id 是 sp|ID|Name，尝试取 ID
                    if '|' in s_id:
                        short_id = s_id.split('|')[1]
                        found_train_label = train_labels_map.get(short_id)
                    
                    # 尝试 2: 暴力模糊搜索 (如果训练集不大，这很快)
                    if found_train_label is None:
                         for train_id in train_labels_map:
                             if s_id in train_id or train_id in s_id:
                                 found_train_label = train_labels_map[train_id]
                                 break
                
                if found_train_label is not None:
                    # 预测类别 = Hit 的训练集类别
                    pred_lbl = found_train_label
                    
                    # 计算连续分数 (用于 AUPRC/AUROC)
                    # 逻辑: 确信的正样本分最高，确信的负样本分最低
                    safe_eval = e_val + 1e-200
                    log_score = -np.log10(safe_eval) # E值越小，log_score 越大
                    
                    if pred_lbl == 1:
                        score = log_score       # 正数
                    else:
                        score = -log_score      # 负数
                else:
                    # Hit 到了但找不到 Label (罕见)
                    pred_lbl = 0
                    score = 0.0
            
            y_pred.append(pred_lbl)
            y_scores.append(score)

        # 6. 计算当前种子的指标
        metrics = calculate_full_metrics(y_true, y_pred, y_scores)
        metrics['Seed'] = seed
        all_metrics.append(metrics)
        print(f"   [Seed {seed}] MCC: {metrics.get('MCC', 0):.4f} | AUPRC: {metrics.get('AUPRC', 0):.4f}")

        # 清理临时文件
        for ext in ['.phr', '.pin', '.psq', '.pdb', '.pot', '.ptf', '.pto', '.nhr', '.nin', '.nsq', '.txt']:
             f_path = db_name + ext if 'txt' not in ext else out_file
             if os.path.exists(f_path): os.remove(f_path)

    # 汇总统计
    if all_metrics:
        df = pd.DataFrame(all_metrics)
        cols = ['Seed', 'Sens.', 'Spec.', 'MCC', 'AUROC', 'AUPRC', 'Acc', 'F1', 'Prec']
        df = df[cols]
        
        summary = df.drop(columns='Seed').agg(['mean', 'std'])
        
        print("\n" + "="*80)
        print("📊 BLAST Baseline Results (Internal Test Set) - Mean ± SD:")
        print(summary)
        print("="*80)
        
        df.to_csv(OUTPUT_FILE, index=False)
        summary.to_csv(OUTPUT_FILE.replace('.csv', '_summary.csv'))
        print(f"Detailed results saved to {OUTPUT_FILE}")
    else:
        print("No metrics calculated.")

if __name__ == "__main__":
    main()

🚀 Starting BLAST Baseline using Pickle Labels (Seeds: [42, 2023, 1234, 567, 100])...

>>> Processing Seed 42...
   Loaded 274 train labels and 69 test labels.
   [Seed 42] MCC: 0.7557 | AUPRC: 0.9028

>>> Processing Seed 2023...
   Loaded 276 train labels and 67 test labels.
   [Seed 2023] MCC: 0.7084 | AUPRC: 0.7588

>>> Processing Seed 1234...
   Loaded 275 train labels and 68 test labels.
   [Seed 1234] MCC: 0.6982 | AUPRC: 0.9158

>>> Processing Seed 567...
   Loaded 275 train labels and 68 test labels.
   [Seed 567] MCC: 0.7954 | AUPRC: 0.8855

>>> Processing Seed 100...
   Loaded 275 train labels and 68 test labels.
   [Seed 100] MCC: 0.7809 | AUPRC: 0.8113

📊 BLAST Baseline Results (Internal Test Set) - Mean ± SD:
         Sens.     Spec.       MCC     AUROC     AUPRC       Acc        F1  \
mean  0.875088  0.897109  0.747717  0.929391  0.854843  0.891166  0.821101   
std   0.049227  0.032425  0.043110  0.008557  0.067225  0.019774  0.030371   

          Prec  
mean  0.776037  


In [8]:
import os
import torch
import torch.nn as nn
import pickle
import pandas as pd
import numpy as np
from sklearn.metrics import (accuracy_score, roc_auc_score, confusion_matrix, 
                             precision_score, f1_score, matthews_corrcoef, 
                             average_precision_score)

# ================= Configuration =================
SEEDS = [42, 2023, 1234, 567, 100]

# 指向 Script 1 中定义的保存目录
MODEL_DIR = "./saved_models_2026_eval/"

# 测试数据文件 
TEST_DATA_FILES = [
    "dataset_easy_redundant_features.pkl" 
]

DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
FIXED_ENCODING = "prot_t5"
INPUT_DIM = 1024 
# ===========================================

class FinalSelectionMLP(nn.Module):
    def __init__(self, input_dim=1024):
        super(FinalSelectionMLP, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.5), 
            nn.Linear(512, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.5),     
            nn.Linear(256, 64),  nn.ReLU(),                                         
            nn.Linear(64, 1)                                                       
        )
    def forward(self, x): return self.net(x)

def calculate_metrics(y_true, y_probs):
    y_pred = (y_probs >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    
    metrics = {
        'Acc': accuracy_score(y_true, y_pred),
        'Sn': tp / (tp + fn) if (tp + fn) > 0 else 0,
        'Sp': tn / (tn + fp) if (tn + fp) > 0 else 0,
        'Prec': precision_score(y_true, y_pred, zero_division=0),
        'F1': f1_score(y_true, y_pred, zero_division=0),
        'MCC': matthews_corrcoef(y_true, y_pred)
    }
    
    try:
        if len(np.unique(y_true)) > 1:
            metrics['AUC'] = roc_auc_score(y_true, y_probs)
            metrics['AUPRC'] = average_precision_score(y_true, y_probs)
        else:
            metrics['AUC'] = 0.5
            metrics['AUPRC'] = 0.0
    except:
        metrics['AUC'] = 0.0
        metrics['AUPRC'] = 0.0
        
    return metrics

def evaluate_models_independently(X_features, y_true, input_dim):
    """
    对 5 个模型分别进行预测和评估，记录各自的指标。
    """
    all_seed_metrics = []
    individual_probs = {}
    
    print(f">>> Evaluating {len(SEEDS)} models independently...")
    
    for seed in SEEDS:
        model_path = os.path.join(MODEL_DIR, f"model_seed_{seed}.pth")
        scaler_path = os.path.join(MODEL_DIR, f"scaler_seed_{seed}.pkl")
        
        if not os.path.exists(model_path) or not os.path.exists(scaler_path):
            print(f"❌ Missing model or scaler for seed {seed}, skipping...")
            continue
            
        # 加载 Scaler 并标准化数据
        with open(scaler_path, 'rb') as f:
            scaler = pickle.load(f)
        X_scaled = scaler.transform(X_features)
        X_tensor = torch.FloatTensor(X_scaled).to(DEVICE)
        
        # 加载模型
        model = FinalSelectionMLP(input_dim=input_dim).to(DEVICE)
        model.load_state_dict(torch.load(model_path, map_location=DEVICE))
        model.eval()
        
        # 预测与评估
        with torch.no_grad():
            logits = model(X_tensor)
            probs = torch.sigmoid(logits).cpu().numpy().flatten()
            individual_probs[f'prob_seed_{seed}'] = probs
            
            # 计算当前随机种子模型的独立指标
            metrics = calculate_metrics(y_true, probs)
            metrics['Seed'] = seed
            all_seed_metrics.append(metrics)
            
    return all_seed_metrics, individual_probs

def main():
    for pkl_file in TEST_DATA_FILES:
        if not os.path.exists(pkl_file): 
            print(f"Skipping missing file: {pkl_file}")
            continue
            
        print(f"\n{'='*80}")
        print(f"Evaluating Dataset: {pkl_file}")
        
        with open(pkl_file, 'rb') as f:
            data = pickle.load(f)
        
        ids = [item['id'] for item in data]
        features = np.stack([item[FIXED_ENCODING] for item in data])
        y_true = np.array([item['label'] for item in data])
        
        print(f"Data Loaded: {len(data)} samples")

        # --- 核心：分别评估并收集指标 ---
        all_seed_metrics, individual_probs = evaluate_models_independently(features, y_true, INPUT_DIM)
        
        if not all_seed_metrics:
            print("Failed to generate predictions.")
            continue
            
        # --- 计算 Mean 和 SD ---
        # 提取所有的 metric keys (排除 'Seed')
        metric_keys = [k for k in all_seed_metrics[0].keys() if k != 'Seed']
        summary_stats = {}
        
        for k in metric_keys:
            values = [m[k] for m in all_seed_metrics]
            # ddof=1 表示计算样本标准差，这在论文汇报中是最标准的做法
            summary_stats[k] = {
                'mean': np.mean(values),
                'std': np.std(values, ddof=1) 
            }

        print(f"\n[Performance Summary across {len(all_seed_metrics)} Seeds]")
        print("-" * 100)
        # 输出你想要的 Mean ± SD 格式
        print(f"Acc: {summary_stats['Acc']['mean']:.4f} ± {summary_stats['Acc']['std']:.4f} | "
              f"MCC: {summary_stats['MCC']['mean']:.4f} ± {summary_stats['MCC']['std']:.4f} | "
              f"AUC: {summary_stats['AUC']['mean']:.4f} ± {summary_stats['AUC']['std']:.4f} | "
              f"AUPRC: {summary_stats['AUPRC']['mean']:.4f} ± {summary_stats['AUPRC']['std']:.4f}")
        
        print(f"Sn : {summary_stats['Sn']['mean']:.4f} ± {summary_stats['Sn']['std']:.4f} | "
              f"Sp : {summary_stats['Sp']['mean']:.4f} ± {summary_stats['Sp']['std']:.4f}")
        print("-" * 100)
        
        # 1. 保存每次 Seed 的详细指标到 CSV，方便你直接复制到论文表格
        df_metrics = pd.DataFrame(all_seed_metrics)
        # 增加一行作为 Mean
        mean_row = {k: summary_stats[k]['mean'] for k in metric_keys}
        mean_row['Seed'] = 'Mean'
        # 增加一行作为 SD
        std_row = {k: summary_stats[k]['std'] for k in metric_keys}
        std_row['Seed'] = 'SD'
        
        df_metrics = pd.concat([df_metrics, pd.DataFrame([mean_row, std_row])], ignore_index=True)
        metrics_csv = "metrics_summary.csv"
        df_metrics.to_csv(metrics_csv, index=False)
        print(f"Metrics summary saved to: {metrics_csv}")
        
        # 2. 保存每个样本的各个模型预测概率
        result_dict = {'id': ids, 'true_label': y_true}
        result_dict.update(individual_probs)
        df_res = pd.DataFrame(result_dict)
        probs_csv = "predictions_raw.csv"
        df_res.to_csv(probs_csv, index=False)

if __name__ == "__main__":
    main()


Evaluating Dataset: ../cat_2026/dataset_easy_redundant_features.pkl
Data Loaded: 812 samples
>>> Evaluating 5 models independently...


/tmp/ipykernel_92932/4163193708.py:89: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path, map_location=DEVICE))
/tmp/ipykernel_92932/


[Performance Summary across 5 Seeds]
----------------------------------------------------------------------------------------------------
Acc: 0.9276 ± 0.0069 | MCC: 0.7664 ± 0.0221 | AUC: 0.9351 ± 0.0136 | AUPRC: 0.8719 ± 0.0088
Sn : 0.8195 ± 0.0180 | Sp : 0.9529 ± 0.0053
----------------------------------------------------------------------------------------------------
Metrics summary saved to: metrics_summary.csv


In [ ]:
import os
import pandas as pd
import numpy as np
import subprocess
import pickle
from sklearn.metrics import (matthews_corrcoef, accuracy_score, confusion_matrix, 
                             precision_score, recall_score, f1_score, 
                             roc_auc_score, average_precision_score)

# ================= Configuration Area =================
SEEDS = [42, 2023, 1234, 567, 100]
TRAIN_DIR_TEMPLATE = "split_seed_{}"    # Your training set folder path template
TRAIN_FILE_NAME = "train.fasta"          # Training set filename
TEST_FASTA = "dataset_easy_redundant.fasta"           # Your independent test set
TEST_LABEL_FILE = "labeled_sequences_for_encoding.csv" # Test set ground truth

# Output filename
SUMMARY_FILE = "blast_baseline_full_metrics.csv"
# ===========================================

def run_command(cmd):
    """Execute Shell Command"""
    process = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    stdout, stderr = process.communicate()
    if process.returncode != 0:
        print(f"Error executing command: {cmd}")
        print(stderr.decode())

def get_train_labels(fasta_path):
    """Load training labels from the corresponding pickle file"""
    pkl_path = os.path.dirname(fasta_path) + "/train_features.pkl"
    if os.path.exists(pkl_path):
        with open(pkl_path, 'rb') as f:
            data = pickle.load(f)
        return {item['id']: item['label'] for item in data}
    else:
        print(f"Warning: {pkl_path} not found, cannot retrieve training labels!")
        return {}

def calculate_full_metrics(y_true, y_pred, y_scores):
    """Calculate comprehensive metrics including AUC/AUPRC"""
    # Confusion Matrix
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    
    # Basic Metrics
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0) # Sensitivity
    f1 = f1_score(y_true, y_pred, zero_division=0)
    mcc = matthews_corrcoef(y_true, y_pred)
    
    # Specificity
    sp = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    # Advanced Metrics (Require Scores)
    try:
        if len(np.unique(y_true)) > 1:
            auc = roc_auc_score(y_true, y_scores)
            auprc = average_precision_score(y_true, y_scores)
        else:
            auc, auprc = 0.5, 0.0
    except ValueError:
        auc, auprc = 0.0, 0.0

    return {
        "Acc": acc, "Sn": rec, "Sp": sp, "Prec": prec, 
        "F1": f1, "MCC": mcc, "AUC": auc, "AUPRC": auprc
    }

def main():
    # 1. Load Test Labels
    if not os.path.exists(TEST_LABEL_FILE):
        print("Please provide the test label file path.")
        return
    test_df = pd.read_csv(TEST_LABEL_FILE)
    test_truth = dict(zip(test_df['Entry_ID'], test_df['Label']))
    
    all_metrics = []

    print(f"Starting 5-fold BLAST Baseline Testing with Full Metrics...")

    for seed in SEEDS:
        print(f"\n>>> Processing Seed {seed}...")
        folder = TRAIN_DIR_TEMPLATE.format(seed)
        train_fasta = os.path.join(folder, TRAIN_FILE_NAME)
        
        if not os.path.exists(train_fasta):
            print(f"Skipping: {train_fasta} not found")
            continue

        # A. Make DB
        db_name = f"temp_db_seed_{seed}"
        # Only log errors
        run_command(f"makeblastdb -in {train_fasta} -dbtype prot -out {db_name} -logfile /dev/null")

        # B. BLAST Search
        out_file = f"blast_out_seed_{seed}.txt"
        # Output format 6: qseqid sseqid evalue ...
        cmd_blast = f"blastp -query {TEST_FASTA} -db {db_name} -out {out_file} -outfmt \"6 qseqid sseqid evalue\" -max_target_seqs 1 -evalue 1e-3"
        run_command(cmd_blast)

        # C. Parse Results
        train_labels = get_train_labels(train_fasta)
        
        blast_hits = {}
        blast_evalues = {}

        if os.path.exists(out_file) and os.path.getsize(out_file) > 0:
            # Read relevant columns: qseqid, sseqid, evalue
            df_res = pd.read_csv(out_file, sep='\t', header=None, names=['qseqid', 'sseqid', 'evalue'])
            # Keep top hit
            df_res = df_res.sort_values('evalue').drop_duplicates(subset='qseqid', keep='first')
            
            blast_hits = dict(zip(df_res['qseqid'], df_res['sseqid']))
            blast_evalues = dict(zip(df_res['qseqid'], df_res['evalue']))

        y_true = []
        y_pred = []   # Binary predictions (0/1)
        y_scores = [] # Continuous scores for AUC (-log(evalue))

        for q_id, true_lbl in test_truth.items():
            y_true.append(true_lbl)
            
            # Default values (No Hit)
            pred_lbl = 0
            score = 0.0 

            if q_id in blast_hits:
                s_id = blast_hits[q_id]
                e_val = blast_evalues[q_id]
                
                # Retrieve label from training set
                if s_id in train_labels:
                    found_label = train_labels[s_id]
                    
                    pred_lbl = found_label
                    
                    log_score = -np.log10(e_val + 1e-200) # Avoid log(0)
                    if pred_lbl == 1:
                        score = log_score
                    else:
                        score = -log_score # Push negative hits to the bottom of the ranking
                else:
                    # Rare case: ID not in pickle
                    pred_lbl = 0
                    score = 0.0
            else:
                # No significant hit found -> Predicted as Negative (or background)
                pred_lbl = 0
                score = 0.0 # Zero score implies "uncertain/negative" relative to a strong positive hit

            y_pred.append(pred_lbl)
            y_scores.append(score)

        # D. Calculate All Metrics
        metrics = calculate_full_metrics(y_true, y_pred, y_scores)
        metrics['Seed'] = seed
        all_metrics.append(metrics)
        
        print(f"   [Seed {seed}] MCC: {metrics['MCC']:.4f} | Acc: {metrics['Acc']:.4f} | AUC: {metrics['AUC']:.4f}")
        
        # Cleanup
        for ext in ['.phr', '.pin', '.psq', '.pdb', '.pot', '.ptf', '.pto', '.txt']:
            try:
                if os.path.exists(db_name + ext): os.remove(db_name + ext)
            except: pass
        if os.path.exists(out_file): os.remove(out_file)

    # Summary
    df = pd.DataFrame(all_metrics)
    # Move Seed to first column
    cols = ['Seed'] + [c for c in df.columns if c != 'Seed']
    df = df[cols]
    
    # Calculate Mean +/- SD
    summary_stats = df.drop(columns='Seed').agg(['mean', 'std'])
    
    print("\n" + "="*80)
    print("BLAST Baseline Final Results (Mean ± SD):")
    print("-" * 80)
    header = " | ".join([f"{col:^8}" for col in summary_stats.columns])
    print(f"Metric:  {header}")
    print("-" * 80)
    
    mean_row = " | ".join([f"{val:^8.4f}" for val in summary_stats.loc['mean']])
    std_row =  " | ".join([f"{val:^8.4f}" for val in summary_stats.loc['std']])
    
    print(f"Mean  :  {mean_row}")
    print(f"Std   :  {std_row}")
    print("="*80)
    
    df.to_csv(SUMMARY_FILE, index=False)
    print(f"Detailed results saved to {SUMMARY_FILE}")

if __name__ == "__main__":
    main()

In [101]:
import os
import pandas as pd
import numpy as np
import subprocess
import pickle
from sklearn.metrics import (matthews_corrcoef, accuracy_score, confusion_matrix, 
                             precision_score, recall_score, f1_score, 
                             roc_auc_score, average_precision_score)

# ================= Configuration Area =================
SEEDS = [42, 2023, 1234, 567, 100]
TRAIN_DIR_TEMPLATE = "split_seed_{}"    # Your training set folder path template
TRAIN_FILE_NAME = "train.fasta"          # Training set filename
TEST_FASTA = "dataset_easy_redundant.fasta"           # Your independent test set
TEST_LABEL_FILE = "labeled_sequences_for_encoding.csv" # Test set ground truth

# Output filename
SUMMARY_FILE = "blast_baseline_full_metrics.csv"
# ===========================================

def run_command(cmd):
    """Execute Shell Command"""
    process = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    stdout, stderr = process.communicate()
    if process.returncode != 0:
        print(f"Error executing command: {cmd}")
        print(stderr.decode())

def get_train_labels(fasta_path):
    """Load training labels from the corresponding pickle file"""
    pkl_path = os.path.dirname(fasta_path) + "/train_features.pkl"
    if os.path.exists(pkl_path):
        with open(pkl_path, 'rb') as f:
            data = pickle.load(f)
        return {item['id']: item['label'] for item in data}
    else:
        print(f"Warning: {pkl_path} not found, cannot retrieve training labels!")
        return {}

def calculate_full_metrics(y_true, y_pred, y_scores):
    """Calculate comprehensive metrics including AUC/AUPRC"""
    # Confusion Matrix
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    
    # Basic Metrics
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0) # Sensitivity
    f1 = f1_score(y_true, y_pred, zero_division=0)
    mcc = matthews_corrcoef(y_true, y_pred)
    
    # Specificity
    sp = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    # Advanced Metrics (Require Scores)
    try:
        if len(np.unique(y_true)) > 1:
            auc = roc_auc_score(y_true, y_scores)
            auprc = average_precision_score(y_true, y_scores)
        else:
            auc, auprc = 0.5, 0.0
    except ValueError:
        auc, auprc = 0.0, 0.0

    return {
        "Acc": acc, "Sn": rec, "Sp": sp, "Prec": prec, 
        "F1": f1, "MCC": mcc, "AUC": auc, "AUPRC": auprc
    }

def main():
    # 1. Load Test Labels
    if not os.path.exists(TEST_LABEL_FILE):
        print("Please provide the test label file path.")
        return
    test_df = pd.read_csv(TEST_LABEL_FILE)
    test_truth = dict(zip(test_df['Entry_ID'], test_df['Label']))
    
    all_metrics = []

    print(f"Starting 5-fold BLAST Baseline Testing with Full Metrics...")

    for seed in SEEDS:
        print(f"\n>>> Processing Seed {seed}...")
        folder = TRAIN_DIR_TEMPLATE.format(seed)
        train_fasta = os.path.join(folder, TRAIN_FILE_NAME)
        
        if not os.path.exists(train_fasta):
            print(f"Skipping: {train_fasta} not found")
            continue

        # A. Make DB
        db_name = f"temp_db_seed_{seed}"
        # Only log errors
        run_command(f"makeblastdb -in {train_fasta} -dbtype prot -out {db_name} -logfile /dev/null")

        # B. BLAST Search
        out_file = f"blast_out_seed_{seed}.txt"
        # Output format 6: qseqid sseqid evalue ...
        cmd_blast = f"blastp -query {TEST_FASTA} -db {db_name} -out {out_file} -outfmt \"6 qseqid sseqid evalue\" -max_target_seqs 1 -evalue 1e-3"
        run_command(cmd_blast)

        # C. Parse Results
        train_labels = get_train_labels(train_fasta)
        
        blast_hits = {}
        blast_evalues = {}

        if os.path.exists(out_file) and os.path.getsize(out_file) > 0:
            # Read relevant columns: qseqid, sseqid, evalue
            df_res = pd.read_csv(out_file, sep='\t', header=None, names=['qseqid', 'sseqid', 'evalue'])
            # Keep top hit
            df_res = df_res.sort_values('evalue').drop_duplicates(subset='qseqid', keep='first')
            
            blast_hits = dict(zip(df_res['qseqid'], df_res['sseqid']))
            blast_evalues = dict(zip(df_res['qseqid'], df_res['evalue']))

        y_true = []
        y_pred = []   # Binary predictions (0/1)
        y_scores = [] # Continuous scores for AUC (-log(evalue))

        for q_id, true_lbl in test_truth.items():
            y_true.append(true_lbl)
            
            # Default values (No Hit)
            pred_lbl = 0
            score = 0.0 

            if q_id in blast_hits:
                s_id = blast_hits[q_id]
                e_val = blast_evalues[q_id]
                
                # Retrieve label from training set
                if s_id in train_labels:
                    found_label = train_labels[s_id]
                    
                    pred_lbl = found_label
                    
                    log_score = -np.log10(e_val + 1e-200) # Avoid log(0)
                    if pred_lbl == 1:
                        score = log_score
                    else:
                        score = -log_score # Push negative hits to the bottom of the ranking
                else:
                    # Rare case: ID not in pickle
                    pred_lbl = 0
                    score = 0.0
            else:
                # No significant hit found -> Predicted as Negative (or background)
                pred_lbl = 0
                score = 0.0 # Zero score implies "uncertain/negative" relative to a strong positive hit

            y_pred.append(pred_lbl)
            y_scores.append(score)

        # D. Calculate All Metrics
        metrics = calculate_full_metrics(y_true, y_pred, y_scores)
        metrics['Seed'] = seed
        all_metrics.append(metrics)
        
        print(f"   [Seed {seed}] MCC: {metrics['MCC']:.4f} | Acc: {metrics['Acc']:.4f} | AUC: {metrics['AUC']:.4f}")
        
        # Cleanup
        for ext in ['.phr', '.pin', '.psq', '.pdb', '.pot', '.ptf', '.pto', '.txt']:
            try:
                if os.path.exists(db_name + ext): os.remove(db_name + ext)
            except: pass
        if os.path.exists(out_file): os.remove(out_file)

    # Summary
    df = pd.DataFrame(all_metrics)
    # Move Seed to first column
    cols = ['Seed'] + [c for c in df.columns if c != 'Seed']
    df = df[cols]
    
    # Calculate Mean +/- SD
    summary_stats = df.drop(columns='Seed').agg(['mean', 'std'])
    
    print("\n" + "="*80)
    print("BLAST Baseline Final Results (Mean ± SD):")
    print("-" * 80)
    header = " | ".join([f"{col:^8}" for col in summary_stats.columns])
    print(f"Metric:  {header}")
    print("-" * 80)
    
    mean_row = " | ".join([f"{val:^8.4f}" for val in summary_stats.loc['mean']])
    std_row =  " | ".join([f"{val:^8.4f}" for val in summary_stats.loc['std']])
    
    print(f"Mean  :  {mean_row}")
    print(f"Std   :  {std_row}")
    print("="*80)
    
    df.to_csv(SUMMARY_FILE, index=False)
    print(f"Detailed results saved to {SUMMARY_FILE}")

if __name__ == "__main__":
    main()

Starting 5-fold BLAST Baseline Testing with Full Metrics...

>>> Processing Seed 42...
   [Seed 42] MCC: 0.8609 | Acc: 0.9569 | AUC: 0.9110

>>> Processing Seed 2023...
   [Seed 2023] MCC: 0.8494 | Acc: 0.9532 | AUC: 0.9124

>>> Processing Seed 1234...
   [Seed 1234] MCC: 0.8054 | Acc: 0.9421 | AUC: 0.8827

>>> Processing Seed 567...
   [Seed 567] MCC: 0.8507 | Acc: 0.9544 | AUC: 0.8942

>>> Processing Seed 100...
   [Seed 100] MCC: 0.8353 | Acc: 0.9495 | AUC: 0.8891

BLAST Baseline Final Results (Mean ± SD):
--------------------------------------------------------------------------------
Metric:    Acc    |    Sn    |    Sp    |   Prec   |    F1    |   MCC    |   AUC    |  AUPRC  
--------------------------------------------------------------------------------
Mean  :   0.9512  |  0.8623  |  0.9720  |  0.8787  |  0.8699  |  0.8403  |  0.8979  |  0.8220 
Std   :   0.0058  |  0.0414  |  0.0037  |  0.0105  |  0.0191  |  0.0215  |  0.0133  |  0.0100 
Detailed results saved to blast_baseli

In [6]:
import os
import subprocess

# ================= 配置区域 =================
# 你可以只跑一个特定的 Seed (例如 [42])，也可以跑全部
SEEDS = [42, 2023, 1234, 567, 100]  
FOLDER_TEMPLATE = "split_seed_{}"    # 你的文件夹命名规则
# ===========================================

def run_command(cmd):
    """执行 Shell 命令并捕获异常"""
    try:
        print(f"执行命令: {cmd}")
        # 这里去掉了 stdout/stderr 的屏蔽，方便你看到 BLAST 的实时进度或报错
        subprocess.run(cmd, shell=True, check=True)
    except subprocess.CalledProcessError as e:
        print(f"❌ 命令执行失败: {cmd}")
        print(f"错误信息: {e}")

def main():
    print("🚀 开始生成 BLAST 结果文件...\n")

    for seed in SEEDS:
        folder = FOLDER_TEMPLATE.format(seed)
        
        train_fasta = os.path.join(folder, "train.fasta")
        test_fasta = os.path.join("dataset_easy_redundant.fasta")
        
        # 检查 FASTA 文件是否存在
        if not os.path.exists(train_fasta) or not os.path.exists(test_fasta):
            print(f"⚠️ 跳过 Seed {seed}: 找不到 {train_fasta} 或 {test_fasta}")
            continue

        print(f"[{folder}] 正在处理...")
        
        # 1. 建立 BLAST 数据库
        db_name = f"temp_db_seed_{seed}"
        cmd_makedb = f"makeblastdb -in {train_fasta} -dbtype prot -out {db_name}"
        run_command(cmd_makedb)

        # 2. 运行 BLASTP 搜索
        # -max_target_seqs 1: 只保留最相似的一条
        # -evalue 10: 宽松阈值，确保能抓到远源同源序列的分数
        # -outfmt "6 qseqid sseqid evalue": 只输出我们需要的 3 列信息
        out_file = f"blast_out_seed_{seed}.txt"
        cmd_blastp = f"blastp -query {test_fasta} -db {db_name} -out {out_file} -outfmt \"6 qseqid sseqid evalue\" -max_target_seqs 1 -evalue 10 -num_threads 8"
        run_command(cmd_blastp)

        # 3. 清理 BLAST 产生的临时数据库文件 (保持文件夹整洁)
        print(f"[{folder}] 清理临时数据库文件...")
        for ext in ['.phr', '.pin', '.psq', '.pdb', '.pot', '.ptf', '.pto', '.nhr', '.nin', '.nsq']:
            temp_file = db_name + ext
            if os.path.exists(temp_file):
                os.remove(temp_file)

        print(f"✅ Seed {seed} 完成！结果已保存至同级目录下的: {out_file}\n")
        print("-" * 60)

if __name__ == "__main__":
    main()

🚀 开始生成 BLAST 结果文件...

[split2_seed_42] 正在处理...
执行命令: makeblastdb -in split2_seed_42/train.fasta -dbtype prot -out temp_db_seed_42


Building a new DB, current time: 03/26/2026 21:53:04
New DB name:   /home/gaozhw/voltage_model_classification/sequence/first_seq/temp_db_seed_42
New DB title:  split2_seed_42/train.fasta
Sequence type: Protein
Keep MBits: T
Maximum file size: 1000000000B
Adding sequences from FASTA; added 274 sequences in 0.00786996 seconds.
执行命令: blastp -query ../cat_2026/dataset_easy_redundant.fasta -db temp_db_seed_42 -out blast_out_seed_42.txt -outfmt "6 qseqid sseqid evalue" -max_target_seqs 1 -evalue 10 -num_threads 8
[split2_seed_42] 清理临时数据库文件...
✅ Seed 42 完成！结果已保存至同级目录下的: blast_out_seed_42.txt

------------------------------------------------------------
[split2_seed_2023] 正在处理...
执行命令: makeblastdb -in split2_seed_2023/train.fasta -dbtype prot -out temp_db_seed_2023


Building a new DB, current time: 03/26/2026 21:53:09
New DB name:   /home/gaozhw/voltage_model_cla

In [7]:
import pandas as pd
import numpy as np
import os
import pickle
from collections import defaultdict

# ================= 配置区域 =================
SEEDS = [42, 2023, 1234, 567, 100]

# 1. DL 模型预测结果
DL_PRED_FILE = "prediction_results_dataset_easy_redundant_features.csv"

# 2. BLAST 输出文件模板 (注意根据你上一个脚本的运行位置，它应该在当前目录)
BLAST_OUT_TEMPLATE = "blast_out_seed_{}.txt"

# 3. 对应种子的训练集标签文件模板 (用于获取 sseqid 的真实类别)
TRAIN_PKL_TEMPLATE = "split_seed_{}/train_features.pkl"

# 4. 输出文件
OUTPUT_CSV = "DL_wins_Ensemble_BLAST_fails_2026.csv"
# ===========================================

def load_train_labels(pkl_path):
    """加载特定种子的训练集标签"""
    if not os.path.exists(pkl_path):
        print(f"⚠️ 找不到标签文件: {pkl_path}")
        return {}
    
    with open(pkl_path, 'rb') as f:
        data = pickle.load(f)
        
    labels_map = {}
    if isinstance(data, list):
        for item in data:
            seq_id = str(item.get('id', item.get('Entry_ID', '')))
            if seq_id:
                labels_map[seq_id] = int(item.get('label', item.get('Label', 0)))
    elif isinstance(data, dict):
        for k, v in data.items():
            labels_map[str(k)] = int(v)
    return labels_map

def main():
    print("🔍 开始进行 5-Seed BLAST 集成与 DL 模型终极对比...\n")

    # 1. 加载 DL 预测结果作为主数据框架
    if not os.path.exists(DL_PRED_FILE):
        print(f"❌ 找不到 DL 结果文件: {DL_PRED_FILE}")
        return
    
    df_dl = pd.read_csv(DL_PRED_FILE)
    df_dl['id'] = df_dl['id'].astype(str)
    # 创建一个字典方便查询: {id: {'true_label': x, 'pred_label': y, 'pred_prob': z}}
    dl_data = {}
    for _, row in df_dl.iterrows():
        dl_data[str(row['id'])] = {
            'true_label': int(row['true_label']),
            'pred_label': int(row['pred_label']),
            'pred_prob': float(row['pred_prob'])
        }
    print(f"✅ 成功加载 DL 预测结果，共 {len(dl_data)} 条序列。")

    # 2. 收集每个种子的 BLAST 预测结果
    # 结构: blast_votes[query_id] = [seed1_pred, seed2_pred, ...]
    blast_votes = defaultdict(list)
    
    for seed in SEEDS:
        blast_file = BLAST_OUT_TEMPLATE.format(seed)
        pkl_file = TRAIN_PKL_TEMPLATE.format(seed)
        
        if not os.path.exists(blast_file) or not os.path.exists(pkl_file):
            print(f"⚠️ 缺失 Seed {seed} 的 BLAST 结果或标签文件，跳过...")
            continue
            
        train_labels = load_train_labels(pkl_file)
        
        # 解析当前的 BLAST 输出
        df_blast = pd.read_csv(blast_file, sep='\t', header=None, names=['qseqid', 'sseqid', 'evalue'])
        df_blast = df_blast.sort_values('evalue').drop_duplicates(subset='qseqid', keep='first')
        
        # 记录当前种子的预测字典
        current_seed_preds = {}
        for _, row in df_blast.iterrows():
            q_id = str(row['qseqid'])
            s_id = str(row['sseqid'])
            
            # 匹配标签
            train_lbl = train_labels.get(s_id)
            if train_lbl is None and '|' in s_id:
                train_lbl = train_labels.get(s_id.split('|')[1])
                
            if train_lbl is not None:
                current_seed_preds[q_id] = train_lbl
                
        # 将当前种子的预测加入到总投票箱中 (没匹配到的默认记为 0)
        for q_id in dl_data.keys():
            pred = current_seed_preds.get(q_id, 0)
            blast_votes[q_id].append(pred)
            
        print(f"   - Seed {seed} 结果已并入投票箱 (有效 Hit 数: {len(current_seed_preds)}).")

    # 3. 多数投票汇总并比较
    cases = []
    
    for q_id, info in dl_data.items():
        votes = blast_votes[q_id]
        
        # 如果没有足够的种子数据，跳过
        if len(votes) == 0:
            continue
            
        # 多数投票：大于等于总票数一半则为 1
        sum_votes = sum(votes)
        final_blast_pred = 1 if sum_votes >= (len(votes) / 2.0) else 0
        
        true_lbl = info['true_label']
        dl_pred = info['pred_label']
        dl_prob = info['pred_prob']
        
        # 场景 A: DL 对了 (VGIC)，而 BLAST 集成预测错了 (判断为非VGIC)
        if true_lbl == 1 and dl_pred == 1 and final_blast_pred == 0:
            cases.append({
                'ID': q_id,
                'True_Label': true_lbl,
                'DL_Pred': dl_pred,
                'DL_Prob': round(dl_prob, 4),
                'Ensemble_BLAST_Pred': final_blast_pred,
                'BLAST_Vote_Details': f"{sum_votes} out of {len(votes)} voted VGIC",
                'Advantage_Type': 'Remote Homolog (BLAST ensemble missed it)'
            })
            
        # 场景 B: DL 排除了假阳性，而 BLAST 被相似度骗了
        elif true_lbl == 0 and dl_pred == 0 and final_blast_pred == 1:
             cases.append({
                'ID': q_id,
                'True_Label': true_lbl,
                'DL_Pred': dl_pred,
                'DL_Prob': round(dl_prob, 4),
                'Ensemble_BLAST_Pred': final_blast_pred,
                'BLAST_Vote_Details': f"{sum_votes} out of {len(votes)} voted VGIC",
                'Advantage_Type': 'Avoided BLAST Trap (False Positive by BLAST)'
            })

    # 4. 保存和输出
    df_cases = pd.DataFrame(cases)
    if not df_cases.empty:
        df_cases = df_cases.sort_values(by=['Advantage_Type', 'DL_Prob'], ascending=[True, False])
        df_cases.to_csv(OUTPUT_CSV, index=False)
        
        print("\n🎉 成功提取出 DL 优于 Ensemble BLAST 的高置信度案例！")
        print("各类优势统计:")
        print(df_cases['Advantage_Type'].value_counts())
        print(f"👉 详细结果已保存至: {OUTPUT_CSV}")
        
        print("\n🔥 推荐写进论文的明星序列 (Top 3 远源同源):")
        top_remote = df_cases[df_cases['Advantage_Type'] == 'Remote Homolog (BLAST ensemble missed it)'].head(3)
        for _, row in top_remote.iterrows():
            print(f"   - UniProt ID: {row['ID']} (DL Prob: {row['DL_Prob']:.4f}) | BLAST 投票分布: {row['BLAST_Vote_Details']}")
    else:
        print("\n🧐 没有找到 DL 完全赢过 BLAST 的集成案例。")

if __name__ == "__main__":
    main()

🔍 开始进行 5-Seed BLAST 集成与 DL 模型终极对比...

✅ 成功加载 DL 预测结果，共 812 条序列。
   - Seed 42 结果已并入投票箱 (有效 Hit 数: 812).
   - Seed 2023 结果已并入投票箱 (有效 Hit 数: 812).
   - Seed 1234 结果已并入投票箱 (有效 Hit 数: 811).
   - Seed 567 结果已并入投票箱 (有效 Hit 数: 812).
   - Seed 100 结果已并入投票箱 (有效 Hit 数: 811).

🎉 成功提取出 DL 优于 Ensemble BLAST 的高置信度案例！
各类优势统计:
Advantage_Type
Avoided BLAST Trap (False Positive by BLAST)    24
Remote Homolog (BLAST ensemble missed it)        5
Name: count, dtype: int64
👉 详细结果已保存至: DL_wins_Ensemble_BLAST_fails_2026.csv

🔥 推荐写进论文的明星序列 (Top 3 远源同源):
   - UniProt ID: Q92952 (DL Prob: 0.8521) | BLAST 投票分布: 2 out of 5 voted VGIC
   - UniProt ID: P58391 (DL Prob: 0.6776) | BLAST 投票分布: 2 out of 5 voted VGIC
   - UniProt ID: Q00195 (DL Prob: 0.6540) | BLAST 投票分布: 0 out of 5 voted VGIC
